In [23]:
# oktibbeha county fips code: 105
import geopandas as gpd
import rasterio as rio
from rasterio.mask import mask
from shapely.geometry import Polygon
import numpy as np
from rasterio.transform import xy
from shapely.geometry import Point
from src.mslandcover.config import LEGEND_CLASSES

ms_places = gpd.read_file('./data/shapefiles/tl_2024_28_place')

okt_confidence_raster_path = r"Z:\guser\dh\MS_HiRes_LC_Prelim\105\lc_confidence_clipped.tif"
with rio.open(okt_confidence_raster_path) as src:
    okt_confidence_profile = src.profile

places_of_interest = ms_places.loc[ms_places['NAME'].isin(['Starkville', 'Mississippi State'])].to_crs(okt_confidence_profile['crs'])

places_of_interest = places_of_interest.dissolve()
geom = Polygon(places_of_interest['geometry'].iloc[0].exterior.coords)

with rio.open(okt_confidence_raster_path) as src:
    lc_confidence, out_transform = mask(src, [geom], crop=True, all_touched=True)
    out_profile = src.profile
    out_profile.update(
        height=lc_confidence.shape[1],
        width=lc_confidence.shape[2],
        transform=out_transform
    )
    
with rio.open(r"G:\postprocessing_tests\okt_confidence_clipped.tif", 'w', **out_profile) as dst:
    dst.write(lc_confidence)

okt_classes_raster_path = okt_confidence_raster_path.replace('confidence', 'classes')
with rio.open(okt_classes_raster_path) as src:
    lc_classes, out_transform = mask(src, [geom], crop=True, all_touched=True)
    out_profile = src.profile
    out_profile.update(
        height=lc_classes.shape[1],
        width=lc_classes.shape[2],
        transform=out_transform
    )
    cmap = src.colormap(1)

with rio.open(r"G:\postprocessing_tests\okt_classes_clipped.tif", 'w', **out_profile) as dst:
    dst.write(lc_classes)
    dst.write_colormap(1, cmap)

okt_probs_raster_path = okt_confidence_raster_path.replace('confidence', 'probs')
with rio.open(okt_probs_raster_path) as src:
    lc_probs, out_transform = mask(src, [geom], crop=True, all_touched=True)
    out_profile = src.profile
    out_profile.update(
        height=lc_probs.shape[1],
        width=lc_probs.shape[2],
        transform=out_transform
    )

with rio.open(r"G:\postprocessing_tests\okt_probs_clipped.tif", 'w', **out_profile) as dst:
    dst.write(lc_probs)

lc_confidence = lc_confidence.squeeze()
lc_classes = lc_classes.squeeze()
lc_probs = lc_probs.squeeze()

In [ ]:
# randomly sample points where confidence is low

np.random.seed(1701)

n_samples = 200
max_confidence = 25

# randomly sample points where confidence is below threshold and class is not 0
pop_points = np.argwhere((lc_confidence < max_confidence) & (lc_classes != 0))
sample_indices = np.random.choice(range(pop_points.shape[0]), n_samples, replace=False)
sample_points = pop_points[sample_indices]

# get predicted classes for each sample point
sample_classes = lc_classes[sample_points[:, 0], sample_points[:, 1]]
sample_class_names = [LEGEND_CLASSES[cls] for cls in sample_classes]

# convert sample points to coordinates in raster crs
sample_coords = [Point(xy(out_transform, point[0], point[1])) for point in sample_points]


sampled_points_gdf = gpd.GeoDataFrame({
    'confidence': lc_confidence[sample_points[:, 0], sample_points[:, 1]],
    'pred_class_index': sample_classes,
    'pred_class_name': sample_class_names,
},geometry=sample_coords, crs=places_of_interest.crs)
sampled_points_gdf.to_file(r"G:\postprocessing_tests\sampled_points.gpkg")

[[ 3520 11204]
 [ 3277  2529]
 [ 5389  8267]
 [ 8576  1249]
 [ 6307  6479]
 [ 6409  4477]
 [ 5358  5794]
 [ 5597  4107]
 [ 3982  4521]
 [ 6039  7946]
 [ 6977  3747]
 [ 3673  7844]
 [ 9722  5796]
 [ 5041  2637]
 [ 3195  7314]
 [ 4250  9057]
 [ 8852  1610]
 [ 6895  4432]
 [ 2427  8496]
 [ 3825 11566]
 [ 6207  8136]
 [ 6554  5888]
 [ 7061  3567]
 [ 6449  4004]
 [ 5673  6211]
 [ 4994  5808]
 [ 9403  2041]
 [ 6277  5173]
 [ 7610  3078]
 [ 4897  4447]
 [ 5220  4870]
 [ 2872  1549]
 [ 3105  5571]
 [ 1881  5915]
 [ 3689  4696]
 [ 6972  5318]
 [ 6283  6400]
 [ 9868  6103]
 [ 2618  9631]
 [ 2359  8497]
 [ 1332  4041]
 [ 3982  1612]
 [ 6312  4548]
 [ 5606  3936]
 [ 4774  5931]
 [ 6059  8005]
 [ 5827  6006]
 [ 7970  5847]
 [ 1375  6014]
 [ 4541  6117]
 [ 6997  7314]
 [ 4482  5618]
 [ 3381  9397]
 [ 4976  7901]
 [ 8526  7108]
 [ 2633  5759]
 [ 9971  6037]
 [ 3302  5489]
 [ 9168  1178]
 [ 5211  6162]
 [ 7735  5269]
 [ 5045  9771]
 [ 4613  7398]
 [ 4587  7596]
 [ 7470  7095]
 [ 2354  4658]
 [ 6429  4

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.windows import from_bounds
import numpy as np
import os

# Input and reference rasters
input_raster = r"G:\NAIP_MS_2023\ortho_1-1_hc_s_ms105_2023_1\ortho_1-1_hc_s_ms105_2023_1.tif"
reference_raster = r"G:\postprocessing_tests\okt_classes_clipped.tif"
final_raster = r"G:\postprocessing_tests\MSU_OKT_0.1M_GSD.tif"

# Target GSD (resolution in meters per pixel)
target_xres = 0.1
target_yres = 0.1

# Step 1: Reproject the input raster to the reference raster's CRS
with rasterio.open(reference_raster) as ref:
    dst_crs = ref.crs  # Reference CRS
    ref_bounds = ref.bounds  # Reference bounds (used for clipping)
    ref_transform = ref.transform  # Reference transform (used for clipping)

with rasterio.open(input_raster) as src:
    # Calculate the transform and new dimensions for the reprojected raster
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )

    # Update the metadata for the reprojected raster
    reprojected_meta = src.meta.copy()
    reprojected_meta.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height,
    })

    # Create the reprojected raster (temporary location)
    reprojected_raster = "temp_reprojected.tif"
    with rasterio.open(reprojected_raster, "w", **reprojected_meta) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest  # Use nearest for resampling
            )

print(f"Reprojection completed. Saved to: {reprojected_raster}")

# Step 2: Clip and Resample to 0.1m GSD resolution

with rasterio.open(reprojected_raster) as src:
    # Define the clipping window using the reference raster's bounds
    window = from_bounds(*ref_bounds, transform=src.transform)
    clipped_transform = src.window_transform(window)  # Clipped transform
    clipped_width = window.width
    clipped_height = window.height

    # Calculate the output dimensions for 0.1m resolution
    clipped_bounds = src.bounds
    resampled_width = int((clipped_bounds[2] - clipped_bounds[0]) / target_xres)
    resampled_height = int((clipped_bounds[3] - clipped_bounds[1]) / target_yres)

    # New transform for 0.1m GSD resolution
    resampled_transform = rasterio.transform.from_origin(clipped_bounds[0], clipped_bounds[3], target_xres, target_yres)

    # Update metadata for the final raster
    final_meta = src.meta.copy()
    final_meta.update({
        "height": resampled_height,
        "width": resampled_width,
        "transform": resampled_transform,
        "compress": "lzw",
    })

    # Create the final raster with resampling and clipping in one step
    with rasterio.open(final_raster, "w", **final_meta) as dst:
        for i in range(1, src.count + 1):
            src_band = src.read(i, window=window)
            dst_band = np.empty((resampled_height, resampled_width), dtype=src_band.dtype)

            # Resample using Lanczos
            reproject(
                source=src_band,
                destination=dst_band,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=resampled_transform,
                dst_crs=src.crs,
                resampling=Resampling.lanczos
            )

            dst.write(dst_band, i)

print(f"Clipping and resampling completed. Final output saved to: {final_raster}")


os.remove(reprojected_raster)  # Remove the temporary reprojected raster

Reprojection completed. Saved to: temp_reprojected.tif
